# Exercises XP: Minimal MCP over STDIO (Student)

Build a tiny MCP server and client that talk over STDIO. This code is supposed to be executed in a local jupyter notebook not Colab's notebook.

## What you'll learn
- How MCP structures hosts/clients/servers and why STDIO is great locally.
- How to register a tool (action) and a resource (read-only context) on a server.
- How to write a client that initializes, lists, and invokes those features.

## Setup
Run the install cell, then restart the runtime if Colab asks. Python 3.10+ required.

In [ ]:
# Install MCP CLI + SDK
%pip install -qU "mcp[cli]"

In [ ]:
# Quick verify
!python --version
!mcp --help | head -n 5

## A. Server (server.py)
Create a small MCP server named "Demo" with:
- Tool `add(a: int, b: int) -> int` returning the sum.
- Resource template `greeting://{name}` returning "Hello, {name}!".
- Start the STDIO loop in `__main__`.

In [7]:
%%writefile mcp-101/server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Demo")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Return the sum of two integers."""
    return a + b

@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Return a greeting for the given name."""
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting mcp-101/server.py


## B. Client (client.py)
Write a client that:
1) Spawns the server via STDIO using the MCP CLI.
2) Initializes a session.
3) Lists resources and tools, printing their names.
4) Reads `greeting://hello` and prints it.
5) Calls tool `add` with a=1, b=7 and prints the result.

In [1]:
%%writefile mcp-101/client.py
import asyncio

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from pydantic import AnyUrl


# Configure the MCP server process
server_params = StdioServerParameters(
    command="mcp",
    args=["run", "server.py"],
    env=None,
)


async def run():
    # Start the server using STDIO
    async with stdio_client(server_params) as (read, write):

        # Create and initialize the client session
        async with ClientSession(read, write) as session:
            await session.initialize()

            # List available tools
            tools = await session.list_tools()
            print("Available tools:")
            for tool in tools.tools:
                print(f" - {tool.name}")

            # List available resources
            resources = await session.list_resources()
            print("\nAvailable resources:")
            for resource in resources.resources:
                print(f" - {resource.uri}")

            # List resource templates
            templates = await session.list_resource_templates()
            print("\nResource templates:")
            for template in templates.resourceTemplates:
                print(f" - {template.uriTemplate}")

            # Read the greeting resource
            greeting = await session.read_resource(
                AnyUrl("greeting://hello")
            )

            print("\nGreeting result:")
            for content in greeting.contents:
                print(content.text)

            # Call the add tool
            result = await session.call_tool(
                "add",
                arguments={"a": 1, "b": 7},
            )

            print("\nAdd tool result:")
            for content in result.content:
                if content.type == "text":
                    print(content.text)


if __name__ == "__main__":
    asyncio.run(run())


Overwriting mcp-101/client.py


## C. Run
One terminal (client spawns server):
```
python client.py
```

Or two terminals:
```
mcp run server.py
python client.py
```

In Colab, run the next cell (client will spawn the server automatically).

In [2]:
# Run the client (spawns the server over STDIO)
%cd mcp-101
!python client.py

d:\Documentos\Developers Institute Projects\DI_Bootcamp\Week10\Day3\mcp-101
Available tools:
 - add

Available resources:

Resource templates:
 - greeting://{name}

Greeting result:
Hello, hello!

Add tool result:
8


[11/04/26 12:00:00] INFO     Processing request of type           server.py:733
                             ListToolsRequest                                  
                    INFO     Processing request of type           server.py:733
                             ListResourcesRequest                              
                    INFO     Processing request of type           server.py:733
                             ListResourceTemplatesRequest                      
                    INFO     Processing request of type           server.py:733
                             ReadResourceRequest                               
                    INFO     Processing request of type           server.py:733
                             CallToolRequest                                   


## Troubleshooting
- `mcp: command not found` ? rerun the install cell or restart runtime.
- Connection closed ? open a second terminal and run `mcp run server.py` to check server errors.
- Type errors ? ensure JSON args are ints for `add`.